# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
%pip -q install duckdb huggingface_hub

import os, getpass, duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:18} {n:>12,} rows")   # metadata-only — confirms the unit of analysis exists at the right scale

dim_clients                 104 rows
dim_content             519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily           78,835,655 rows
fact_daily_sample    11,694,072 rows
fact_query_90d        2,414,248 rows


**Unit of analysis:** one row = one content item's Search Console performance on one day
(`client_hash_id` + `content_hash_id` + `report_date`) in `fact_content_daily_performance`.

**Time window:** March 2026 — a mid-panel month, not the sealed `_sample` month (June 2026).
Split at the midpoint: days 1–15 = feature window (decision moment), days 16–31 = outcome window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
desc = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
print(desc[["column_name", "column_type"]].to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

**Table(s):** `fact_content_daily_performance`, sliced to `month=2026-03`. `fact_content_query_90d`
is left out — its 90-day trailing window isn't confirmed aligned to "as of March 15," so joining
it here risks a window-alignment leak.

**Label / proxy:** `is_declining` = 1 if summed impressions in days 16–31 < 80% of summed
impressions in days 1–15 — same shape as `w02`'s `trend_direction == "down"`, built fresh since
the warehouse has no pre-computed trend column.

| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id` | Context | pseudonym IDs — join/group only |
| `report_date` | Context | defines the window split, not a model input |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (days 1–15) | Feature | known as of the decision day |
| `gsc_data_available` | Context | filter flag, not a feature |
| `gsc_impressions` (days 16–31 / `impressions_h2`) | Label / proxy | the label is built from it — never a feature |
| `fact_content_query_90d` (all columns) | Excluded | window-alignment not confirmed — see above |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
sample_pair = con.sql(f"""
    SELECT client_hash_id, content_hash_id FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31' LIMIT 1
""").df().iloc[0]

grain_check = con.sql(f"""
    SELECT report_date, COUNT(*) AS rows_on_this_date
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id = '{sample_pair.client_hash_id}'
      AND content_hash_id = '{sample_pair.content_hash_id}'
      AND report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY report_date ORDER BY report_date
""").df()
print(f"rows in March: {len(grain_check)}  |  max rows on any single date: {grain_check['rows_on_this_date'].max()} (should be 1)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows in March: 31  |  max rows on any single date: 1 (should be 1)


Three queries, on `month=2026-03`, then the five-feature frame, then the trap.

In [6]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content_items,n_clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [7]:
bool_cols = desc.loc[desc["column_type"].str.upper() == "BOOLEAN", "column_name"].tolist()
print("Boolean columns found:", bool_cols)

BOOLEAN_COLUMN = "gsc_data_available"
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE {BOOLEAN_COLUMN} IS TRUE) AS available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
avail["pct_available"] = (avail["available_rows"] / avail["total_rows"] * 100).round(1)
avail

Boolean columns found: ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,pct_available
0,9841378,3611061,36.7


In [8]:
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_h1,
           SUM(gsc_clicks) AS clicks_h1,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_h1,
           AVG(gsc_avg_position) AS avg_position_h1,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_h1
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()
print(f"{len(feature_frame):,} content items with a usable feature row")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 content items with a usable feature row


,client_hash_id,content_hash_id,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,0.000000,3.659683,15
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,0.010050,4.086084,13
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,0.002141,4.449176,13
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,0.000000,6.600595,14
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,0.001297,1.883472,14


1. `impressions_h1` — summed daily impressions, days 1–15. Knowable same-day.
2. `clicks_h1` — same reasoning, summed daily clicks.
3. `ctr_h1` — clicks_h1 / impressions_h1. A ratio of two already-knowable sums.
4. `avg_position_h1` — mean daily rank, days 1–15. Logged per report day.
5. `active_days_h1` — count of days with impressions > 0 in the window. A count of days already past.

In [9]:
outcome = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY 1, 2
""").df()
data = feature_frame.merge(outcome, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["impressions_h2"] < 0.8 * data["impressions_h1"]).astype(int)
print(f"{len(data):,} rows | declining rate: {data['is_declining'].mean():.1%}")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
md = data.dropna(subset=honest_features + ["is_declining"])
X_tr, X_te, y_tr, y_te = train_test_split(md[honest_features], md["is_declining"],
                                            test_size=0.25, random_state=42, stratify=md["is_declining"])
honest_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST score — AUC: {honest_auc:.3f}")

# THE TRAP: add the label-derived column on purpose
leaky_features = honest_features + ["impressions_h2"]
ld = data.dropna(subset=leaky_features + ["is_declining"])
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(ld[leaky_features], ld["is_declining"],
                                                test_size=0.25, random_state=42, stratify=ld["is_declining"])
leaky_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f"LEAKY score — AUC: {leaky_auc:.3f}  <- jumps toward 1.0  (+{leaky_auc - honest_auc:.3f})")

# REMOVE THE LEAK, KEEP THE HONEST NUMBER
del leaky_features
print(f"Final reported score (honest features only): AUC {honest_auc:.3f}")
print("impressions_h2 excluded — it is the quantity the label threshold is built from.")

import json, os
os.makedirs("../outputs", exist_ok=True)
json.dump({"honest_auc": float(honest_auc), "leaky_auc": float(leaky_auc),
           "n_rows": int(len(data)), "declining_rate": float(data["is_declining"].mean())},
          open("../outputs/w03_contract_metrics.json", "w"), indent=2, sort_keys=True)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 rows | declining rate: 29.6%
HONEST score — AUC: 0.590
LEAKY score — AUC: 1.000  <- jumps toward 1.0  (+0.410)
Final reported score (honest features only): AUC 0.590
impressions_h2 excluded — it is the quantity the label threshold is built from.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
history_spread = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_gsc, MAX(gsc_data_start) AS latest_gsc,
           MIN(ga4_data_start) AS earliest_ga4, MAX(ga4_data_start) AS latest_ga4,
           COUNT(*) AS n_clients
    FROM {TABLES['dim_clients']}
""").df()
history_spread
# cross-reference: Query 3 above already showed what share of March rows are gsc_data_available —
# that percentage is the quantitative face of this same limitation.


,earliest_gsc,latest_gsc,earliest_ga4,latest_ga4,n_clients
0,2025-01-27,2026-06-02,2025-10-29,2026-06-01,104


**Unbalanced history, GSC-only early rows.** `dim_clients.gsc_data_start` / `ga4_data_start`
vary per client, so March 2026 isn't equally "mid-panel" for everyone — and rows before a
client's `ga4_data_start` carry zero-filled GA4 columns rather than true zero engagement
(Haris's point: `0` is a verified result, a missing/flagged row is a collection gap, not the
same thing). This notebook doesn't correct for that difference yet.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.